## Task 1 - Python Data Pipeline
### - Zain Sharief


In [1]:
import hashlib
import json
import logging
import re
import sqlite3
import tempfile
import time
from contextlib import closing
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
SCHEMA_FILE = PROJECT_DIR / "schema.sql"
INITIAL_FILE = PROJECT_DIR / "sales_data.json"
INCREMENTAL_FILE = PROJECT_DIR / "sales_incremental.json"
MAX_SQLITE_INT = 2**63 - 1

if not SCHEMA_FILE.is_file():
    raise FileNotFoundError("Start Jupyter from the extracted project folder.")

## Extract and transform

- Read a UTF-8 JSON array and parse decimal values exactly. Flatten 'product', trim required text, validate business rules, and calculate gross and net sales. 
- A null or missing discount is treated as zero. 
- Date formats are parsed explicitly and stored in UTC. 
- For duplicate IDs within a batch, the first valid row is considered. 
- Every later valid occurrence is logged and skipped. 
- Invalid rows are logged with their source index, transaction ID, and validation reasons.
- Comments have been provided throughout the cells for further explanation.

In [2]:
def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="microseconds").replace("+00:00", "Z")


# Using this function we prepare a logger that the pipeline uses to write to etl.log. It creates the log folder if needed and formats each entry 
# with a UTC timestamp and a severity level. The pipeline will use it to record batch starts and commits, warnings for rejected or duplicate rows, and errors when a batch fails.
def make_logger(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sales_etl")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    handler = logging.FileHandler(path, encoding="utf-8")
    formatter = logging.Formatter("%(asctime)sZ %(levelname)s %(message)s", "%Y-%m-%dT%H:%M:%S")
    formatter.converter = time.gmtime
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# This function is defined to reject invalid numeric values like NaN, infinity etc.
def reject_non_json_number(token):
    raise ValueError(f"Invalid JSON number: {token}")


def extract_json(path):
    with path.open(encoding="utf-8") as source:
        records = json.load(
            source, parse_float=Decimal, parse_constant=reject_non_json_number
        )
    if not isinstance(records, list):
        raise ValueError("Expected a top-level JSON array.")
    return records

#converts the different date formats in the sales JSON into one consistent UTC timestamp before loading into SQLite
#This ensures consistency since every date is stored in the same format, making it easy for reporting
def parse_timestamp(value):
    if not isinstance(value, str):
        raise ValueError("date must be a string")

    value = value.strip()
    formats = (
        (r"[0-9]{4}/[0-9]{2}/[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}", "%Y/%m/%d %H:%M:%S"),
        (r"[0-9]{2}-[0-9]{2}-[0-9]{4}", "%d-%m-%Y"),
        (r"[0-9]{4}-[0-9]{2}-[0-9]{2}", "%Y-%m-%d"),
    )
    try:
        for pattern, date_format in formats:
            if re.fullmatch(pattern, value):
                parsed = datetime.strptime(value, date_format)
                break
        else:
            pattern = (
                r"[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}"
                r"(?:\.[0-9]{1,6})?(?:Z|[+-](?:[01][0-9]|2[0-3]):[0-5][0-9])?"
            )
            if not re.fullmatch(pattern, value):
                raise ValueError("Unsupported timestamp format.")
            parsed = datetime.fromisoformat(value)

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        return parsed.astimezone(timezone.utc).isoformat(
            timespec="microseconds"
        ).replace("+00:00", "Z")
    except (ValueError, OverflowError):
        raise ValueError(f"date is invalid or unsupported: {value!r}") from None


def required_text(value, field, reasons):
    if isinstance(value, str) and value.strip():
        return value.strip()
    reasons.append(f"{field} must be nonblank text")
    return None


# Here scaled_decimal enforces data quality rules for prices and discounts and stores them consistently. 
# It checks that values are within the allowed range and decimal precision
def scaled_decimal(value, field, places, maximum, reasons):
    if type(value) not in (int, Decimal) or not (0 <= value <= maximum):
        reasons.append(f"{field} must be a number between 0 and {maximum}")
        return None
    decimal_value = Decimal(value)
    if decimal_value != decimal_value.quantize(Decimal(1).scaleb(-places)):
        reasons.append(f"{field} can have at most {places} decimal places")
        return None
    return int(decimal_value.scaleb(places))

In [3]:
#define output column order
BUSINESS_COLUMNS = (
    "transaction_id", "customer_id", "product_id", "product_name",
    "product_category", "unit_price_cents", "quantity", "discount_bps",
    "transaction_at", "region", "gross_sales_cents", "net_sales_cents",
)
STAGING_COLUMNS = BUSINESS_COLUMNS + ("row_hash",)

#This is used to clean transactions - flatten, standardize values, collect validation errors, and calculate sales.
#Finally, each of the cleaned rows get a SHA-256 hash.The load step can compare this hash value for the same transaction ID
#to decide whether its values changed.
def transform_record(raw):
    if not isinstance(raw, dict):
        return None, ["record must be an object"]

    reasons = []
    transaction_id = required_text(raw.get("transaction_id"), "transaction_id", reasons)
    customer_id = required_text(raw.get("customer_id"), "customer_id", reasons)
    region = required_text(raw.get("region"), "region", reasons)

    product = raw.get("product")
    if isinstance(product, dict):
        product_id = required_text(product.get("id"), "product.id", reasons)
        product_name = required_text(product.get("name"), "product.name", reasons)
        product_category = required_text(product.get("category"), "product.category", reasons)
        unit_price_cents = scaled_decimal(
            product.get("price"), "product.price", 2, Decimal(MAX_SQLITE_INT).scaleb(-2), reasons
        )
    else:
        reasons.append("product must be an object")

    quantity = raw.get("quantity")
    if type(quantity) is not int or not (0 < quantity <= MAX_SQLITE_INT):
        reasons.append("quantity must be a positive integer within SQLite's integer range")

    discount = raw.get("discount")
    discount_bps = scaled_decimal(0 if discount is None else discount, "discount", 4, 1, reasons)

    try:
        transaction_at = parse_timestamp(raw.get("date"))
    except ValueError as error:
        reasons.append(str(error))

    if reasons:
        return None, reasons

    gross_sales_cents = unit_price_cents * quantity
    if gross_sales_cents > MAX_SQLITE_INT:
        return None, ["gross sales exceeds SQLite's integer range"]
    net_sales_cents = (gross_sales_cents * (10000 - discount_bps) + 5000) // 10000

    values = (
        transaction_id, customer_id, product_id, product_name, product_category,
        unit_price_cents, quantity, discount_bps, transaction_at, region,
        gross_sales_cents, net_sales_cents,
    )
    canonical = json.dumps(values, ensure_ascii=True, separators=(",", ":"))
    row_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return values + (row_hash,), []


#processes the whole file - it calls transform_record() to handle 3 cases -
#Invalid records, valid record with an already processed ID, valid record with a new ID in this file
def transform_batch(records, source_name, logger):
    rows = {}
    rejected = duplicates = 0
    for index, raw in enumerate(records, start=1):
        row, reasons = transform_record(raw)
        if reasons:
            rejected += 1
            record_id = raw.get("transaction_id") if isinstance(raw, dict) else None
            logger.warning(
                "REJECTED source=%s record=%d transaction_id=%r reasons=%s",
                source_name, index, record_id, " | ".join(reasons),
            )
        elif row[0] in rows:
            duplicates += 1
            conflict = row[-1] != rows[row[0]][-1]
            logger.warning(
                "DUPLICATE source=%s record=%d transaction_id=%r conflicting_values=%s",
                source_name, index, row[0], conflict,
            )
        else:
            rows[row[0]] = row

    return list(rows.values()), rejected, duplicates

## Load incrementally

validated records in a temporary (staging) SQLite table. Compare transaction_id and the SHA-256 hash of normalized business values against the target. Insert new IDs, update changed rows, and leave identical rows untouched. The sale date is never used as a filter, so valid late-arriving records can load. The comparison, upsert, and audit insert run in one transaction.

In [4]:
#load_sales() prepares the database and stages the rows
def load_sales(db_path, rows, source_name, started_at, rows_read, rejected, duplicates):
    db_path.parent.mkdir(parents=True, exist_ok=True)
    columns = ", ".join(STAGING_COLUMNS)
    placeholders = ", ".join("?" for _ in STAGING_COLUMNS)
    updates = ", ".join(f"{column} = excluded.{column}" for column in STAGING_COLUMNS[1:])

    with closing(sqlite3.connect(db_path, timeout=30)) as connection:
        connection.executescript(SCHEMA_FILE.read_text(encoding="utf-8"))

        with connection:
            connection.execute("BEGIN IMMEDIATE")
            #WHERE 0 always evaluates to false, so this creates an empty temporary table with the selected column names
            connection.execute(
                f"CREATE TEMP TABLE staged_sales AS SELECT {columns} FROM sales WHERE 0"
            )
            connection.executemany(f"INSERT INTO staged_sales VALUES ({placeholders})", rows)
            #SQLite counts what will be inserted, updated, or skipped
            inserted, updated, unchanged = connection.execute(
                """
                SELECT COUNT(*) FILTER (WHERE target.transaction_id IS NULL),
                       COUNT(*) FILTER (WHERE target.row_hash <> stage.row_hash),
                       COUNT(*) FILTER (WHERE target.row_hash = stage.row_hash)
                FROM staged_sales AS stage
                LEFT JOIN sales AS target USING (transaction_id)
                """
            ).fetchone()

            loaded_at = utc_now()
            #INSERT .. ON CONFLICT(transaction_id) DO UPDATE handles both inserts and updates in one SQL statement.
            connection.execute(
                f"""
                INSERT INTO sales ({columns}, created_at, updated_at)
                SELECT {columns}, ?, ? FROM staged_sales WHERE 1
                ON CONFLICT(transaction_id) DO UPDATE SET {updates}, updated_at = excluded.updated_at
                WHERE sales.row_hash <> excluded.row_hash
                """,
                (loaded_at, loaded_at),
            )
            connection.execute(
                """
                INSERT INTO etl_runs (
                    source_file, started_at, finished_at, rows_read, inserted,
                    updated, unchanged, duplicates, rejected
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (source_name, started_at, utc_now(), rows_read, inserted,
                 updated, unchanged, duplicates, rejected),
            )

    return {
        "source_file": source_name, "rows_read": rows_read, "inserted": inserted,
        "updated": updated, "unchanged": unchanged, "duplicates": duplicates, "rejected": rejected,
    }

#run_batch() records the start time and writes a START message to the log. 
#It reads the JSON, runs the transformation and validation, and passes the clean rows and counts to load_sales(). 
#If an error occurs, logger.exception() records the failure
def run_batch(path, db_path, logger):
    started_at = utc_now()
    logger.info("START source=%s", path.name)
    try:
        records = extract_json(path)
        rows, rejected, duplicates = transform_batch(records, path.name, logger)
        summary = load_sales(
            db_path, rows, path.name, started_at, len(records), rejected, duplicates
        )
    except (OSError, ValueError, ArithmeticError, sqlite3.Error):
        logger.exception("FAILED source=%s", path.name)
        raise
    logger.info("COMMITTED %s", json.dumps(summary))
    return summary

## Run the initial load, incremental load, and replay

In this cell, the ETL demo is run three times against the same SQLite database. It shows what happens when you load the initial file, apply an incremental file, and replay that same incremental file.

In [5]:
output_dir = PROJECT_DIR / "output"
output_dir.mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="demo-", dir=output_dir))
db_path = run_dir / "sales.db"
log_path = run_dir / "etl.log"
logger = make_logger(log_path)

steps = [
    ("Initial load", INITIAL_FILE),
    ("Incremental load", INCREMENTAL_FILE),
    ("Incremental replay", INCREMENTAL_FILE),
]
summaries = [
    {"step": step, **run_batch(path, db_path, logger)}
    for step, path in steps
]

display(pd.DataFrame(summaries).drop(columns="source_file").set_index("step"))
print(f"Database: {db_path.relative_to(PROJECT_DIR).as_posix()}")
print(f"Log:      {log_path.relative_to(PROJECT_DIR).as_posix()}")

Database: output/demo-anc39kf3/sales.db
Log:      output/demo-anc39kf3/etl.log


,rows_read,inserted,updated,unchanged,duplicates,rejected
step,,,,,,
Initial load,600,599,0,0,0,1
Incremental load,154,150,1,1,1,1
Incremental replay,154,0,0,152,1,1


## Verify the database

- Query the final transaction count, inspect the corrected and late-arriving rows, and aggregate sales by region. 
- verify the expected load counts and sample transformations for the supplied files.

In [6]:
with closing(sqlite3.connect(db_path)) as connection:
    total_sales = connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0]
    audit_count = connection.execute("SELECT COUNT(*) FROM etl_runs").fetchone()[0]
    examples = pd.read_sql_query(
        """
        SELECT transaction_id, product_name, quantity, discount_bps, transaction_at,
               net_sales_cents
        FROM sales
        WHERE transaction_id IN ('T001', 'T002', 'T004', 'T601')
        ORDER BY transaction_id
        """,
        connection,
    )
    regional_sales = pd.read_sql_query(
        """
        SELECT region, COUNT(*) AS transactions, SUM(quantity) AS units,
               SUM(net_sales_cents) AS net_sales_cents
        FROM sales
        GROUP BY region
        ORDER BY net_sales_cents DESC
        """,
        connection,
    )
    excluded = connection.execute(
        "SELECT COUNT(*) FROM sales WHERE transaction_id IN ('T003', 'T751')"
    ).fetchone()[0]

expected = [(600, 599, 0, 0, 0, 1), (154, 150, 1, 1, 1, 1), (154, 0, 0, 152, 1, 1)]
count_keys = ("rows_read", "inserted", "updated", "unchanged", "duplicates", "rejected")
assert [tuple(run[key] for key in count_keys) for run in summaries] == expected
assert total_sales == 749 and audit_count == 3 and excluded == 0
assert examples.at[0, "net_sales_cents"] == 189998
assert examples.at[1, "discount_bps"] == 0
assert examples.at[2, "quantity"] == 6 and examples.at[2, "net_sales_cents"] == 25449
assert examples.at[3, "transaction_at"] == "2022-12-31T21:30:00.000000Z"
assert int(regional_sales["transactions"].sum()) == total_sales

print(f"{total_sales} sales rows | {audit_count} committed runs | All verification checks passed")
display(examples.set_index("transaction_id"))
display(regional_sales.set_index("region"))

749 sales rows | 3 committed runs | All verification checks passed


,product_name,quantity,discount_bps,transaction_at,net_sales_cents
transaction_id,,,,,
T001,Laptop,2,500,2023-01-15T10:30:00.000000Z,189998
T002,Mouse,5,0,2023-02-10T14:15:00.000000Z,9995
T004,Keyboard,6,1500,2023-04-20T00:00:00.000000Z,25449
T601,Desk,4,0,2022-12-31T21:30:00.000000Z,75600


,transactions,units,net_sales_cents
region,,,
North,188,873,36881105
West,187,936,36287136
East,187,747,30769225
South,187,811,7944624


## Inspect invalid data and duplicates

In [7]:
for entry in log_path.read_text(encoding="utf-8").splitlines():
    if " WARNING " in entry:
        print(entry.split(" WARNING ", 1)[1])

REJECTED source=sales_data.json record=3 transaction_id='T003' reasons=customer_id must be nonblank text | quantity must be a positive integer within SQLite's integer range
REJECTED source=sales_incremental.json record=153 transaction_id='T751' reasons=date is invalid or unsupported: '2023-02-30'
DUPLICATE source=sales_incremental.json record=154 transaction_id='T750' conflicting_values=False
REJECTED source=sales_incremental.json record=153 transaction_id='T751' reasons=date is invalid or unsupported: '2023-02-30'
DUPLICATE source=sales_incremental.json record=154 transaction_id='T750' conflicting_values=False
